In [ ]:
import tensorflow as tf
import numpy as np
import os
import sys

project_root = os.path.abspath(os.path.join(".."))  # One level up from current script
if project_root not in sys.path:
    sys.path.append(project_root)

from keras.src.metrics.accuracy_metrics import accuracy
from src.model_loader import ModelLoader
from src.data_loader import DataLoader
from src.utils import get_class_weigths, get_confusion_matrix, get_classification_report

print(tf.__version__)

# Nettoyage des données bloquante

Dans cette section rapide, puisque nous avons uniquement des images, nous voulons vérifier que celle-ci sont utilisables pour nos modèles. Ces fonctions permettent de tester si l'image est exploitable, dans le cas contraire nous la supprimons afin que nos tests puissent se dérouler sans accrocs.

In [ ]:
def is_valid_image(path):
    try:
        img_bytes = tf.io.read_file(path)
        decoded_img = tf.io.decode_image(img_bytes)
        return True
    except tf.errors.InvalidArgumentError as e:
        print(f"Found bad path {path}...{e}")
        return False

def clean_invalid_images(datasets_base_path):
    for root, dirs, files in os.walk(datasets_base_path):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                image_path = os.path.join(root, file)
                if not is_valid_image(image_path):
                    print(f"Removing invalid image: {image_path}")
                    os.remove(image_path)

clean_invalid_images("../datasets")

# **Choix des datasets d’entrainements**
Pour l’entraînement et l’évaluation des modèles de classification d’images, deux types de tâches ont été considérés : la classification binaire et la classification multi-classes. Ces tâches ont été testées à l’aide de quatre configurations de jeux de données, différenciées par leur méthode de gestion du déséquilibre entre les classes.
Dans un premier temps sois classification binaire, différenciation entre les images de type Photo (classe positive) et les autres types (Painting, Text, Schematics, Sketch) regroupés comme classe négative ou classification multi-classes, attribution d’une image à l’une des cinq classes.

Puis ensuite le déséquilibre des classes les données initiales présentent un déséquilibre entre les différentes classes, certaines étant sur-représentées. Afin d’atténuer ce biais, deux approches ont été mises en œuvre.

La première consiste à équilibrer les classes en réduisant leur nombre à celui de la classe minoritaire, par échantillonnage aléatoire, complété par de la data augmentation pour enrichir la diversité sans créer de déséquilibre supplémentaire. Bien que cette méthode garantisse une représentation équivalente des classes, elle peut entraîner une perte d’information en raison de la réduction des classes majoritaires. La seconde approche repose sur la pondération des classes dans la fonction de perte, où aucun échantillonnage n’est effectué et toutes les données sont conservées. Les poids attribués à chaque classe, en fonction de leur fréquence, permettent de compenser le déséquilibre en donnant plus d’importance aux classes sous-représentées. Bien que cette méthode utilise toutes les données disponibles, elle est sensible à la définition des poids et peut devenir instable sur des jeux de données de petite taille.

Nous avons donc 4 datasets différents pour tester les 3 modèles que nous avons choisis, un modèle CNN hard, un ResNet, et un InceptionV3. Ces configurations nous permettent de comparer les performances des modèles en fonction des approches d’équilibrage choisies et d'évaluer leur robustesse face au déséquilibre des classes.


In [ ]:
# Préparation des datasets
data_loader = DataLoader()

datasets = {
    "binary_nocw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"]
    ),
    "binary_cw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
    "multiclass_nocw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"]
    ),
    "multiclass_cw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
}

# CNN_HARD

**(Convolutional Neural Network classique, version "hard")**

Ce modèle est un CNN classique, profond et personnalisable, construit couche par couche sans utiliser de réseaux pré-entraînés. Il utilise plusieurs couches Conv2D avec un nombre croissant de filtres (32, 64, 128), suivies de MaxPooling permettant de se concentrer sur les informations importantes de l'image. À la fin, les données sont aplaties (Flatten) puis traitées par des couches Dense pour la classification.
Idéal pour les petits jeux de données ou comme base de comparaison. Il est simple à contrôler, mais peut manquer de performance sans gros entraînement. Il nous sert de base dans nos tests du meilleur model.


In [ ]:
cnn_hard_loader = ModelLoader(model_name="CNN_HARD")

history_cnn_all_ds = {}
all_model_cnn = {}

with tf.device("/gpu:0"):
    for dataset_name, (train_data, val_data, test_data) in datasets.items():
        if 'binary' in dataset_name:
            cnn_hard_model = cnn_hard_loader.create_model_CNN_hard(show_summary=False)
            all_model_cnn[f"{dataset_name}"] = cnn_hard_model
        elif 'multiclass' in dataset_name:
            cnn_hard_model = cnn_hard_loader.create_model_CNN_hard(show_summary=False, num_classes=5)
            all_model_cnn[f"{dataset_name}"] = cnn_hard_model

        if "nocw" in dataset_name:
            history_cnn = cnn_hard_model.fit(
                train_data,
                validation_data=val_data,
                epochs=10,
                verbose=2,
                callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_loader.get_early_stopping(), cnn_hard_loader.get_model_checkpoint()],
            )
            history_cnn_all_ds[f"{dataset_name}"] = history_cnn
        else:
            history_cnn = cnn_hard_model.fit(
                train_data,
                validation_data=val_data,
                epochs=10,
                verbose=2,
                class_weight=get_class_weigths(train_data, val_data),
                callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_loader.get_early_stopping(), cnn_hard_loader.get_model_checkpoint()],
            )
            history_cnn_all_ds[f"{dataset_name}"] = history_cnn

In [ ]:
print(history_cnn_all_ds['multiclass_cw'].history)

# RES_NET

**(Residual Network - ResNet50)**

ResNet, développé par Microsoft en 2015, a remporté l'ImageNet (ILSVRC) en introduisant une idée révolutionnaire : les connexions résiduelles (skip connections). Elles permettent de propager le gradient même dans des réseaux très profonds, évitant le problème de la dégradation.
Le modèle ResNet50 contient 50 couches profondes, dont beaucoup de "bottleneck blocks" organisés en blocs résiduels.
Nous testons ce model car c’est un modèle excellent pour les tâches complexes et les grandes datasets, tout en restant stable à l'entraînement. Très utilisé en pratique grâce à sa robustesse et ses performances.


<img alt="Courbe accuracy CNN" height="400" src="/Users/tanguydumontier/PycharmProjects/CESI_DS/content/SchemaArchi/ResNet50_binaire.png" width="400"/>

In [ ]:
res_net_loader = ModelLoader(model_name="RES_NET")

history_resnet_all_ds = {}
all_model_resnet = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    if 'binary' in dataset_name:
        res_net_model = res_net_loader.create_model_resnet50(show_summary=False)
        all_model_resnet[f"{dataset_name}"] = res_net_model
    elif 'multiclass' in dataset_name:
        res_net_model = res_net_loader.create_model_resnet50(show_summary=False, num_classes=5)
        all_model_resnet[f"{dataset_name}"] = res_net_model

    if "nocw" in dataset_name:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds[f"{dataset_name}"] = history_resnet
    else:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds[f"{dataset_name}"] = history_resnet

# INCEPTION


InceptionV3 est une évolution du réseau GoogLeNet (InceptionV1) proposé par Google. Ce modèle repose sur le principe d’extraction multi-échelle : chaque bloc Inception combine plusieurs convolutions parallèles (1x1, 3x3, 5x5, etc.), ce qui permet au modèle de capturer différentes tailles de motifs simultanément.
InceptionV3 améliore l'efficacité du réseau avec la factorisation des convolutions (ex: un 5x5 remplacé par deux 3x3, ou un 7x7 par un 1x7 + 7x1), réduisant les coûts computationnels sans perte de précision.
Très performant pour les tâches de classification visuelle sur des images complexes. Il est aussi plus léger que d'autres gros modèles tout en restant très précis. C’est pour ça que nous avons choisi de le tester.


In [ ]:

inception_loader = ModelLoader(model_name="INCEPTION")

history_inception_all_ds = {}
all_model_inception = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    if 'binary' in dataset_name:
        inception_model = inception_loader.create_model_with_inception(show_summary=False)
        all_model_inception[f"{dataset_name}"] = inception_model
    elif 'multiclass' in dataset_name:
        inception_model = inception_loader.create_model_with_inception(show_summary=False, num_classes=5)
        all_model_inception[f"{dataset_name}"] = inception_model

    if "nocw" in dataset_name:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            callbacks=[inception_model.get_tensorboard_callback(), inception_model.get_early_stopping(), inception_model.get_model_checkpoint()],
        )
        history_inception_all_ds[f"{dataset_name}"] = history_inception
    else:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[inception_loader.get_tensorboard_callback(), inception_loader.get_early_stopping(), inception_loader.get_model_checkpoint()],
        )
        history_inception_all_ds[f"{dataset_name}"] = history_inception

# **Courbes loss / accuracy**


**Courbe accuracy CNN - Binaire sans classe weight**  
<img src="./../content/Courbes/CNN_binaireNocw.png" width="400"/>

**Courbe accuracy CNN - Binaire avec classe weight**  
<img src="/content/Courbes/CNN_binaireCw.jpg" width="400"/>

**Courbe accuracy CNN - Multiclasse sans classe weight**  
<img src="/content/Courbes/CNN_mutliNocw.jpg" width="400"/>

**Courbe accuracy CNN - Multiclasse avec classe weight**  
<img src="/content/Courbes/CNN_mutliCw.jpg" width="400"/>

**Courbe accuracy ResNet - Binaire sans classe weight**  
<img src="/content/Courbes/ResNet_binaire.jpg" width="400"/>

**Courbe accuracy ResNet - Binaire avec classe weight**  
<img src="/content/Courbes/ResNet_binaireCw.jpg" width="400"/>

**Courbe accuracy ResNet - Multiclasse sans classe weight**  
<img src="/content/Courbes/ResNet_mutli.jpg" width="400"/>

**Courbe accuracy ResNet - Multiclasse avec classe weight**  
<img src="/content/Courbes/ResNet_mutliCw.jpg" width="400"/>

**Courbe accuracy Inception - Binaire sans classe weight**  
<img src="/content/Courbes/Inception_binaire.jpg" width="400"/>

**Courbe accuracy Inception - Binaire avec classe weight**  
<img src="/content/Courbes/Inception_binaireCw.jpg" width="400"/>

**Courbe accuracy Inception - Multiclasse sans classe weight**  
<img src="/content/Courbes/Inception_Mutli.jpg" width="400"/>

**Courbe accuracy Inception - Multiclasse avec classe weight**  
<img src="/content/Courbes/Inception_multiCw.jpg" width="400"/>


# Prédictions et tests des modèles

Afin d'approfondir notre étude sur les différents modèles entraînés, nous passons à la phase de prédictions pour valider nos premières hypothèses. Avec les indicateurs suivants : F1Score, Recall, Precision ainsi que la matrice de confusion, il sera plus simple pour nous de visualiser quel modèle montre les meilleures performances.

## CNN_HARD

### Classification Report

### Confusion Matrix

## RES_NET

### Classification Report

### Confusion Matrix

## INCEPTION

### Classification Report

![CR_Inception1](/Users/tanguydumontier/PycharmProjects/CESI_DS/figures/INCEPTION_classification_report_20250411-165915.png)

### Confusion Matrix

Avec ces graphiques, nous pouvons remarquer que les modèles se basant sur un modèle pré-entraîné sont meilleurs que notre modèle CNN_HARD, en cela en tout point. Par contre, nous voyons qu'Inception affiche des résultats vraiment au dessus avec une excellente précision sur la totalité des jeux de données. Avec l'aide des matrices de confusion nous estimons que la configuration optimale dans notre cas serait ce modèle avec un jeu de donnée binaire avec l'utilisation de poids pour les classes. Il serait toutefois pertinent pour nous de conserver les multi-classes pour le futur, il sera peut-être intéressant de le conserver en guise de comparaison pour les prochaines étapes.

# **Analyse des resultats**

| Modèle       | Test                             | Temps d'exécution (min) | Val_Accuracy (Entraînement) | Val_Loss (Entraînement) | Accuracy (Test) |
|--------------|----------------------------------|---------------------------|------------------------------|---------------------------|------------------|
| **CNN**      | Binaire                          | 4.15             | 0.859                  | 0.312                     | **0.82**         |
| **CNN**      | Binaire + Classe Weight          | 9.42            | 0.850             | 0.364                     | **0.80**         |
| **CNN**      | Multiclasse                      | 2.02               | 0.849           | 0.399                     | **0.78**         |
| **CNN**      | Multiclasse + Classe Weight      | 9.34             | 0.854              | 0.380            | **0.79**         |
| **ResNet**   | Binaire                          | 7.93              | 0.790                  | 0.418              | **0.74**         |
| **ResNet**   | Binaire + Classe Weight          | 16.53              | 0.735                  | 0.422              | **0.71**         |
| **ResNet**   | Multiclasse                      | 3.35              | 0.737                  | 0.624                | **0.68**         |
| **ResNet**   | Multiclasse + Classe Weight      | 14.40              | 0.760                | 0.575               | **0.70**         |
| **Inception**| Binaire                          | 5.55            | 0.965                | 0.090                 | **0.92**         |
| **Inception**| Binaire + Classe Weight          | 17.45               | 0.971                 | 0.087                | **0.93**         |
| **Inception**| Multiclasse                      | 2.90               | 0.958                 | 0.150                  | **0.90**         |
| **Inception**| Multiclasse + Classe Weight      | 17.41             | 0.967                | 0.101                     | **0.91**         |


Au vu des valeurs obtenues, nous avons choisi d’utiliser le modèle Inception avec un jeu de données binaire, en appliquant classe weight. Nous observons que cette configuration affiche les meilleures valeurs d’accuracy et surtout de loss, bien plus faibles que celles des autres modèles. Cela signifie que le modèle apprend efficacement tout en conservant une bonne capacité de généralisation, sans surapprentissage. Ces résultats font de cette combinaison le meilleur compromis pour notre tâche de classification.

# **Methode d'amelioration**

Nous utilisons plusieurs méthodes pour améliorer le compromis biais/variance dans nos modèles. Tout d'abord, le dropout est activé avec un taux de 0.5, ce qui consiste à désactiver aléatoirement 50 % des neurones pendant l'entraînement pour éviter que le modèle ne se suradapte aux données d'entraînement. Nous avons également mis en place le early stopping, qui arrête l'entraînement dès que la performance sur les données de validation cesse de s'améliorer, permettant ainsi de prévenir le surapprentissage. De plus, nous appliquons la data augmentation pour enrichir nos datasets en générant des variations des images d'entraînement, ce qui améliore la capacité du modèle à généraliser. Enfin, nous avons recours à deux techniques de régularisation des données : l'échantillonnage des classes pour équilibrer le nombre d'exemples par classe, et la pondération des classes dans la fonction de perte, qui donne plus d'importance aux classes sous-représentées. Ces méthodes combinées nous permettent de réduire le surapprentissage tout en assurant une bonne généralisation du modèle.